# Assignment IV — demonstration notebook

MBUST · Introduction to Machine Learning · Siman Giri

This notebook does **not** reimplement the network. It imports `code/network.py` and `code/line_follower.py`, runs the same experiments as the brief, and draws the two training curves.

| Task | What this notebook shows |
|---|---|
| 2 | Left-edge line-follower truth table |
| 3.2 | AND, 2–1–1, 1000 epochs, RMSE curve + recall |
| 3.4 | XOR, 2–3–1, 10 000 epochs, RMSE curve + recall |

Seed `7`, $\eta=0.7$, momentum $=0.9$.

In [ ]:
import os
import sys
import random

import matplotlib.pyplot as plt

# notebook lives in Assignment_IV/; modules live in Assignment_IV/code/
HERE = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
CODE = os.path.join(HERE, "code")
if not os.path.isdir(CODE):
    CODE = HERE  # opened from inside code/
sys.path.insert(0, CODE)

from network import Network
from line_follower import controller, WEIGHTS_ARRAY, WEIGHTS

print("code path:", CODE)
print("weights:", WEIGHTS_ARRAY)

## Task 2 — line follower

Weights `{{0.5, 1, -2}, {-0.5, 1, 1}}`. Output is 1 iff the weighted sum $\ge 0$.

In [ ]:
target = {(0, 0): (1, 0), (0, 1): (0, 1), (1, 0): (1, 1), (1, 1): (0, 1)}
print("L  R  |  O1 O2  required")
ok = True
for L, R in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    o1, o2 = controller(L, R)
    req = target[(L, R)]
    match = (o1, o2) == req
    ok = ok and match
    print("%d  %d  |   %d  %d   %s  %s" % (L, R, o1, o2, req, "OK" if match else "MISMATCH"))
print("Table match:", "YES" if ok else "NO")

## Task 3.2 — AND (2–1–1, 1000 epochs)

In [ ]:
random.seed(7)
AND_input = [[0, 0], [1, 0], [0, 1], [1, 1]]
AND_ideal = [[0], [0], [0], [1]]
and_net = Network(2, 1, 1, 0.7, 0.9)

and_epochs, and_rmse = [], []
for epoch in range(1000):
    for x, y in zip(AND_input, AND_ideal):
        and_net.compute_outputs(x)
        and_net.calc_error(y)
        and_net.learn()
    rmse = and_net.get_error(len(AND_input))
    and_epochs.append(epoch + 1)
    and_rmse.append(rmse)
    if (epoch + 1) % 200 == 0 or epoch == 0:
        print("epoch %4d  RMSE = %.6f" % (epoch + 1, rmse))

print("Recall")
for x in AND_input:
    print(x, "->", round(and_net.compute_outputs(x)[0], 4))

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(and_epochs, and_rmse, color="#1B365D", linewidth=1.6)
ax.set_xlabel("Epoch")
ax.set_ylabel("RMSE")
ax.set_title("AND  (2–1–1 MLP, $\\eta=0.7$, momentum $=0.9$)")
ax.set_xlim(1, len(and_epochs))
ax.set_ylim(bottom=0)
ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.7)
fig.tight_layout()
plt.show()

## Task 3.4 — XOR (2–3–1, 10 000 epochs)

This cell takes a few seconds.

In [ ]:
random.seed(7)
xor_input = [[0, 0], [1, 0], [0, 1], [1, 1]]
xor_ideal = [[0], [1], [1], [0]]
xor_net = Network(2, 3, 1, 0.7, 0.9)

xor_epochs, xor_rmse = [], []
for epoch in range(10000):
    for x, y in zip(xor_input, xor_ideal):
        xor_net.compute_outputs(x)
        xor_net.calc_error(y)
        xor_net.learn()
    rmse = xor_net.get_error(len(xor_input))
    xor_epochs.append(epoch + 1)
    xor_rmse.append(rmse)
    if (epoch + 1) % 1000 == 0 or epoch == 0:
        print("epoch %5d  RMSE = %.6f" % (epoch + 1, rmse))

print("Recall")
for x in xor_input:
    print(x, "->", round(xor_net.compute_outputs(x)[0], 4))

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(xor_epochs, xor_rmse, color="#1B365D", linewidth=1.4)
ax.set_xlabel("Epoch")
ax.set_ylabel("RMSE")
ax.set_title("XOR  (2–3–1 MLP, $\\eta=0.7$, momentum $=0.9$)")
ax.set_xlim(1, len(xor_epochs))
ax.set_ylim(bottom=0)
ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.7)
fig.tight_layout()
plt.show()

## Read-off for Task 3.5

1. RMSE falls on both curves; after the first drop it is a slow tail.
2. XOR recalls sit on the correct side of $0.5$ on every vertex.
3. AND is linearly separable (a single threshold unit is enough). XOR is not, so it needs the hidden layer.

The written answers stay in `Assignment_IV.pdf`. This notebook is only the demonstration.